[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/html_scraper.ipynb)

# Scraping a property listings page — the Lecture 2 case study

A saved Mumbai property-listings page: 726 results, ten cards on this page, ten fields per card.
This notebook holds **both versions** of the scraper the lecture walks through — the first draft with
its measured mistakes, then the rewrite — so you can run every claim on the slides yourself.

The page is hosted with the course material; nothing here touches a live site.

## Setup

In [1]:
# !pip install requests beautifulsoup4 pandas lxml

In [2]:
import requests
from bs4 import BeautifulSoup as bs
from copy import copy
import pandas as pd
import re

## Fetch once, parse many times

Download the saved page once, keep it on disk, and parse the local copy from then on
(slide: *Fetch once, parse many times*).

In [3]:
import os

if not os.path.exists("property_listings.html"):
    url = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20collection/data/property_listings.html"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open("property_listings.html", "w", encoding="utf-8") as f:
        f.write(r.text)

with open("property_listings.html", encoding="utf-8") as f:
    soup = bs(f.read(), "lxml")
soup.title.get_text()

'726 11 Property for Sale in Mumbai, Residential Properties | Sulekha Mumbai'

## Mistake 1 — the wrong container

The card's class list is `sk-card listing-card sk-shadow`. Select on the design system's *base*
class and you get three things that are not listings — and the very first one has no `<h2>`.

In [4]:
cards = soup.find_all('div', class_='sk-card')
len(cards)

13

In [5]:
try:
    for card in cards:
        title = card.find('h2').get_text(strip=True)
except AttributeError as e:
    print("crashed on the FIRST card:", e)
    print("that card's classes:", cards[0]['class'])
    print("it holds the page header:", cards[0].h1.get_text(' ', strip=True)[:45], "...")

crashed on the FIRST card: 'NoneType' object has no attribute 'get_text'
that card's classes: ['sk-card', 'sk-shadow']
it holds the page header: 726 Results |            Property for Sale in ...


In [6]:
# the most specific class the container carries — and assert the count
cards = soup.find_all('div', class_='listing-card')
assert len(cards) >= 5, f"only {len(cards)} cards — layout changed?"
len(cards)

10

## The first draft, as written

One hoisted dict, one guarded extraction (the button — the only element never missing),
everything else read bare. Watch where it dies.

In [7]:
property_data = []
listing_info = {}

def draft_body(card):
    listing_info['title']    = card.find('h2').get_text(strip=True)
    listing_info['locality'] = card.find('span', class_='sk-caption-text').get_text(strip=True)
    listing_info['price']    = card.find('strong', class_='rupee').get_text(strip=True)

    # 2 unguarded calls: sk-thumbnail-count is missing on two cards
    listing_info['photos'] = card.find('div', class_='sk-thumbnail-count').span.get_text()

    btn = card.find('div', 'button')
    if btn:                                        # guarded — never needed
        listing_info['ad_id']      = btn.get('data-contentid', "")
        listing_info['advertiser'] = btn.get('data-advtype', "")
    else:
        listing_info['ad_id']      = ''
        listing_info['advertiser'] = ''

    chips_div = card.find('div', class_='sk-chips')
    chips_list = chips_div.find_all('spam', class_='sk-chip')   # <-- 'spam'
    if not chips_list:                                          # always true
        chips_list = chips_div.find_all('div', class_='sk-chip')
    chips = []
    for chip in chips_list:
        chip_text = chip.get_text(strip=True)
        if chip_text not in chips:
            chips.append(chip_text)
    listing_info['chips'] = chips

    key_points = card.find('div', 'prime-highlights scroll-wrap mobile-hide')
    for li in key_points.find_all('li'):
        if 'Config' in li.get_text():
            listing_info['config'] = li.strong.get_text()
        if 'Sale Type' in li.get_text():
            listing_info['sale_type'] = li.strong.get_text()
        if 'Age Of Construction' in li.get_text():
            listing_info['age'] = li.strong.get_text()

    listing_info['seller'] = card.find('div', 'posted').strong.get_text(strip=True)

for i, card in enumerate(cards, start=1):
    try:
        draft_body(card)
    except AttributeError as e:
        print(f"cards 1–{i-1} parsed fine")
        print(f"card {i}   AttributeError: {e}")
        break
    property_data.append(copy(listing_info))

cards 1–4 parsed fine
card 5   AttributeError: 'NoneType' object has no attribute 'span'


**Mistake 2 — guarded in one place, unguarded in the next.** Four good cards were in memory when
card 5 raised; had `to_csv` come after the loop, none would have been written.

Patch just that line and run the whole draft:

In [8]:
property_data = []
listing_info = {}

for card in cards:
    ph = card.find('div', class_='sk-thumbnail-count')          # the one-line guard
    listing_info['photos'] = ph.span.get_text() if ph else None
    try:
        listing_info['title']    = card.find('h2').get_text(strip=True)
        listing_info['locality'] = card.find('span', class_='sk-caption-text').get_text(strip=True)
        listing_info['price']    = card.find('strong', class_='rupee').get_text(strip=True)
        btn = card.find('div', 'button')
        listing_info['ad_id'] = btn.get('data-contentid', "") if btn else ''
        chips_div = card.find('div', class_='sk-chips')
        chips_list = chips_div.find_all('spam', class_='sk-chip')
        if not chips_list:
            chips_list = chips_div.find_all('div', class_='sk-chip')
        listing_info['chips'] = [c.get_text(strip=True) for c in chips_list]
        key_points = card.find('div', 'prime-highlights scroll-wrap mobile-hide')
        for li in key_points.find_all('li'):
            if 'Config' in li.get_text():
                listing_info['config'] = li.strong.get_text()
            if 'Sale Type' in li.get_text():
                listing_info['sale_type'] = li.strong.get_text()
            if 'Age Of Construction' in li.get_text():
                listing_info['age'] = li.strong.get_text()
    except AttributeError as e:
        print("unexpected:", e)
    property_data.append(copy(listing_info))

df_draft = pd.DataFrame(property_data)
print(len(df_draft), "rows")

10 rows


## Mistake 3 — one dictionary, reused ten times

The run "succeeds" now. Look at row 2 — **a five-acre plot of land, listed as a 1 RK resale flat,
10–20 years old.** The plot has no Config / Sale Type / Age rows, so the hoisted dict quietly keeps
the previous flat's values. Rows 3 onward inherit a stale `age` the same way (those flats have a
Possession row instead).

In [9]:
df_draft[['title', 'config', 'sale_type', 'age']].head(5)

,title,config,sale_type,age
0,Flat for Resale in Gaikwad Nagar,1 RK,Resale,10-20 Years
1,Flat for Resale in Malad West,1 RK,Resale,10-20 Years
2,5 Acres Plots & Land for Sale in Andheri,1 RK,Resale,10-20 Years
3,Flat for Sale in Malad West,1 BHK,New,10-20 Years
4,Flat for Sale in Khardi,3 BHK,New,10-20 Years


And the bug the author *did* fix — drop the `copy()` and every row becomes the last card:

In [10]:
rows_nocopy, info = [], {}
for card in cards:
    info['title'] = card.find('h2').get_text(strip=True)
    info['price'] = card.find('strong', class_='rupee').get_text(strip=True)
    rows_nocopy.append(info)          # no copy() — ten references to ONE dict

pd.DataFrame(rows_nocopy).head(3)

,title,price
0,Flat for Sale in Manpada,3.30 Crores
1,Flat for Sale in Manpada,3.30 Crores
2,Flat for Sale in Manpada,3.30 Crores


## Mistake 4 — the typo that never fails, and the string that matches by luck

`find_all('spam', ...)` asks about a tag that does not exist: the answer is `[]`, not an error,
so the fallback silently carries the whole feature. And the spaced class string matches the class
attribute **verbatim** — reorder the classes and it returns `None`.

In [11]:
chips_div = cards[0].find('div', class_='sk-chips')
print("find_all('spam', ...): ", chips_div.find_all('spam', class_='sk-chip'))
print("find_all('div',  ...): ",
      [c.get_text(strip=True) for c in chips_div.find_all('div', class_='sk-chip')])

find_all('spam', ...):  []
find_all('div',  ...):  ['3 Total Floors', '1 bath', 'UnFurnished']


In [12]:
as_is      = bs('<div class="prime-highlights scroll-wrap mobile-hide">x</div>', 'html.parser')
reordered  = bs('<div class="scroll-wrap prime-highlights mobile-hide">x</div>', 'html.parser')

probe = 'prime-highlights scroll-wrap mobile-hide'
print("verbatim string, order as on the page :", as_is.find('div', probe) is not None)
print("verbatim string, classes reordered    :", reordered.find('div', probe) is not None)
print("single class via class_=              :", reordered.find('div', class_='prime-highlights') is not None)

verbatim string, order as on the page : True
verbatim string, classes reordered    : False
single class via class_=              : True


## The rewrite — same logic, three habits

One helper that absorbs the `None` checks, a fresh dict per row, a `try` that costs a row instead
of a run — and label matching that survives capitalisation.

In [13]:
def text_of(el, default=None):
    """Text of an element that may not exist."""
    return el.get_text(' ', strip=True) if el else default

def labelled(card, label):        # the bold value in the matching row
    for li in card.select('.prime-highlights li'):
        first = li.find(string=True)
        if first and first.strip().lower() == label.lower():
            return text_of(li.strong)
    return None

def parse_card(card):
    btn = card.select_one('.button')
    return {                              # fresh dict, every card
        'ad_id':      btn.get('data-contentid') if btn else None,
        'title':      text_of(card.select_one('h2.sk-h6')),
        'locality':   text_of(card.select_one('.location .sk-caption-text')),
        'price':      text_of(card.select_one('.price-info .rupee')),
        'config':     labelled(card, 'config'),
        'sale_type':  labelled(card, 'sale type'),
        'photos':     text_of(card.select_one('.sk-thumbnail-count span')),
        'chips':      '|'.join(text_of(c) for c in card.select('.sk-chip')),
        'advertiser': btn.get('data-advtype') if btn else None,
        'seller':     text_of(card.select_one('.posted strong')),
    }

cards = soup.select('div.listing-card')
assert len(cards) >= 5, f"only {len(cards)} cards"
rows, failed = [], []
for card in cards:
    try:
        rows.append(parse_card(card))
    except Exception as e:
        failed.append(repr(e))
print(f"{len(rows)} parsed, {len(failed)} failed")

df = pd.DataFrame(rows)
df

10 parsed, 0 failed


,ad_id,title,locality,price,config,sale_type,photos,chips,advertiser,seller
0,1002355771,Flat for Resale in Gaikwad Nagar,"Gaikwad Nagar, Mumbai",22 Lakhs,1 RK,Resale,5,3 Total Floors|1 bath|UnFurnished,Owner,by meghna
1,1002355810,Flat for Resale in Malad West,"Malad West, Mumbai",22 Lakhs,1 RK,Resale,9,3 Total Floors|1 bath|UnFurnished,Owner,by meghna
2,1002347847,5 Acres Plots & Land for Sale in Andheri,"Andheri, Mumbai",360 Crores,None,None,1,,Promoter,by Anil
3,1002355263,Flat for Sale in Malad West,"Malad West, Mumbai",50 Lakhs,1 BHK,New,6,21 Total Floors|1 bath|UnFurnished,Broker,by JASH
4,1002355301,Flat for Sale in Khardi,"Khardi, Mumbai",1 Crore,3 BHK,New,None,14 Total Floors|3 bath|Semi Furnished,Owner,by Suresh
5,1002355314,Flat for Sale in New Panvel East,"New Panvel East, Mumbai",1.24 Crore,2 BHK,New,12,7 Total Floors|2 bath|UnFurnished,Builder,by Scarlet builders
6,1002355454,High Rise Apartment for Sale in Goregaon West,"Goregaon West, Mumbai",1.25 Crore,1 BHK,New,11,34 Total Floors|2 bath|UnFurnished,Owner,by Gurtej Oberoi
7,1002346772,Flat for Resale in Abdul Rehman Street,"Abdul Rehman Street, Mumbai",1.68 Crore,1 BHK,Resale,4,Open car park|2 Total Floors|1 bath|Fully Furn...,Owner,by Nik
8,1002355066,Flat for Sale in Malad East,"Malad East, Mumbai",2.25 Crores,2 BHK,New,None,22 Total Floors|2 bath|UnFurnished,Broker,by nayan
9,1002355396,Flat for Sale in Manpada,"Manpada, Mumbai",3.30 Crores,3 BHK,New,13,7 Total Floors|3 bath|UnFurnished,Owner,by Swapnil Navle


Note the plot's row now: `config` and `sale_type` are honest `None`s, its `chips` honestly
empty — visible, countable, nothing invented.

## Everything you scraped is a string

Every column is `object`. Coercion alone would destroy `price` — all ten values carry units —
so the regex lectures cash in here.

In [14]:
df.dtypes

ad_id         object
title         object
locality      object
price         object
config        object
sale_type     object
photos        object
chips         object
advertiser    object
seller        object
dtype: object

In [15]:
print("prices surviving plain to_numeric:",
      pd.to_numeric(df['price'], errors='coerce').notna().sum(), "of", len(df))

prices surviving plain to_numeric: 0 of 10


In [16]:
def to_rupees(s):
    m = re.match(r'([\d.]+)\s*(Lakh|Crore)', s)
    if not m:
        return None
    value, unit = float(m.group(1)), m.group(2)
    return value * (1e5 if unit == 'Lakh' else 1e7)

df['price_inr'] = df['price'].map(to_rupees)
df['photos'] = pd.to_numeric(df['photos'], errors='coerce').astype('Int64')
# ad_id stays a string: 1002355771 is a name, not a quantity
df[['title', 'price', 'price_inr', 'photos']]

,title,price,price_inr,photos
0,Flat for Resale in Gaikwad Nagar,22 Lakhs,2.200000e+06,5
1,Flat for Resale in Malad West,22 Lakhs,2.200000e+06,9
2,5 Acres Plots & Land for Sale in Andheri,360 Crores,3.600000e+09,1
3,Flat for Sale in Malad West,50 Lakhs,5.000000e+06,6
4,Flat for Sale in Khardi,1 Crore,1.000000e+07,<NA>
5,Flat for Sale in New Panvel East,1.24 Crore,1.240000e+07,12
6,High Rise Apartment for Sale in Goregaon West,1.25 Crore,1.250000e+07,11
7,Flat for Resale in Abdul Rehman Street,1.68 Crore,1.680000e+07,4
8,Flat for Sale in Malad East,2.25 Crores,2.250000e+07,<NA>
9,Flat for Sale in Manpada,3.30 Crores,3.300000e+07,13


In [17]:
df.to_csv('property_listings.csv', index=None)
print("written property_listings.csv,", len(df), "rows")

written property_listings.csv, 10 rows


## The scorecard

| # | Finding | Class |
|---|---------|-------|
| 1 | Dict hoisted above the loop — the plot inherited a flat's config | **wrong data** |
| 2 | Stale file silently re-parsed when the fetch fails | **wrong data** |
| 3 | The only guarded extraction was the one never needed | crash |
| 4 | Spaced class string — matches verbatim, dies on reorder | breakage |
| 5 | Labels matched case-sensitively | breakage |
| 6 | `find_all('spam', ...)` — dead code carrying a live feature | dead code |
| 7 | No timeout, no User-Agent, no delay | hygiene |
| 8 | No count assertion (`sk-card` matched 13, not 10) | silence |

**A scraper's most dangerous failure mode is not crashing — it is succeeding with the wrong
numbers.** The full argument is in the Lecture 2 deck, case-study slides.